In [1]:
!pip install polars

In [6]:
import polars as pl
import re

In [7]:
df = pl.read_csv("/content/Customers.csv")

In [8]:
# ---------- 2️⃣ Clean Column Names ----------
cleaned_cols = [
    re.sub(r'[^a-z0-9_]', '', col.strip().lower().replace(' ', '_'))
    for col in df.columns
]
df = df.rename({old: new for old, new in zip(df.columns, cleaned_cols)})

In [9]:
# ---------- 3️⃣ Clean String Columns ----------
for col in df.columns:
    if df[col].dtype == pl.Utf8:
        df = df.with_columns(
            pl.col(col).str.strip_chars().str.to_lowercase().alias(col)
        )

In [10]:
# ---------- 4️⃣ Drop Empty Rows ----------
df = df.drop_nulls(subset=None)

In [11]:
# ---------- 5️⃣ Drop Duplicates ----------
df = df.unique()  # removes duplicate rows
if "email" in df.columns and "phone" in df.columns:
    df = df.unique(subset=["email", "phone"])


In [12]:
# ---------- 6️⃣ Split full_name ----------
if "full_name" in df.columns:
    df = df.with_columns(
        pl.col("full_name").str.split(" ").alias("split_name")
    )
    df = df.with_columns([
        pl.col("split_name").list.get(0).alias("first_name"),
        pl.col("split_name").list.get(1).alias("last_name")
    ])
    df = df.drop("split_name")


In [13]:
# ---------- 7️⃣ Validate Email ----------
if "email" in df.columns:
    df = df.with_columns(
        pl.when(pl.col("email").str.contains("@"))
        .then(pl.col("email"))
        .otherwise(None)
        .alias("email")
    )

In [14]:
# ---------- 8️⃣ Fill Missing Numeric Values ----------
if "income" in df.columns:
    median_income = df["income"].median()
    df = df.with_columns(
        pl.col("income").fill_null(median_income).alias("income")
    )


In [15]:
# ---------- 9️⃣ Save Cleaned Data ----------
df.write_csv("clean_customers.csv")

print("✅ Data cleaning complete and saved as 'clean_customers.csv'")

✅ Data cleaning complete and saved as 'clean_customers.csv'
